In [1]:
import numpy as np
from fig6_utils import *

🎨 Scientific plotting configuration loaded
   Default save format: png
   Contour levels: 100
   Colormap: bwr


In [2]:
import os
from pathlib import Path
p = os.getcwd() + "/../../"
os.chdir(p)
parent_dir = os.getcwd()

In [3]:
config = {
    "years": np.arange(2019, 2025),
    "common_period": np.arange(2004, 2022),
    "imd_folder": f"{parent_dir}/../monsoon-benchmark_data/imd_rainfall_data/4p0", # Path to ground truth rainfall data (4x4 degrees)
    "thresh_file": f"{parent_dir}/../monsoon-benchmark_data/imd_onset_threshold/mwset4x4.nc4",# Path to threshold for the onset of the monsoon (4x4 degrees)
    "thres_file": f"{parent_dir}/../monsoon-benchmark_data/imd_onset_threshold/mwset4x4.nc4",# Path to threshold for the onset of the monsoon (Alternate naming convention)
    "shpfile_path": f"{parent_dir}/../monsoon-benchmark_data/ind_map_shpfile/india_shapefile.shp",  # Path to Shapefile of India
    "output_dir": f"{parent_dir}/examples/paper_figures/outputs",# Directory to save outputs
    "mok": True,
    "day_bins_30": [(1, 5), (6, 10), (11, 15), (16, 20), (21, 25), (26, 30)],
    "day_bins_15": [(1, 5), (6, 10), (11, 15)],
    "mem_num": 51,
    "date_filter_year": 2024,
    "file_pattern": "{}.nc",
    "data_dir": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0",
    "model_paths": {
    "IFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/IFS_S2S",  # IFS model
    "AIFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/AIFS",  # AIFS model
    "FuXi": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi",  # FuXi mdoel
    "Graphcast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GraphCast",  # Graphcast model
    "GenCast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GenCast",  # GenCast model
    "FuXi-S2S": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi_S2S",  # FuXi_S2S model
    "NGCM": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/NeuralGCM",  # NGCM model
}
}


In [ ]:
try:
    skill_fig, gridded_data = generate_fig6(
        config=config
    )
except Exception as e:
    e

Processing 124 years: [1901, 1902, 1903, 1904, 1905, 1906, 1907, 1908, 1909, 1910, 1911, 1912, 1913, 1914, 1915, 1916, 1917, 1918, 1919, 1920, 1921, 1922, 1923, 1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Processing year 1901...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd)

In [ ]:
def package_csv_data_as_grid(csv_data, models_list, lat, lon):
    ret = []

    empty_grid = np.full((8, 9), np.nan)
    empty_da = xr.DataArray(
        data=empty_grid,
        coords={"lat": lat, "lon": lon},
    )

    for val in ["fair_brier_skill", "fair_rps_skill"]:
        for h in csv_data["horizon"].unique():
            loop_list = []
            h_data = csv_data.loc[csv_data["horizon"] == h]

            for model in models_list:
                model_da = empty_da.copy()
                model_data = h_data.loc[h_data["dataset"] == model]

                for ind, row in model_data.iterrows():
                    val_to_assign = row[val]
                    lat_idx = np.where(lat == row["lat"])[0][0]
                    lon_idx = np.where(lon == row["lon"])[0][0]
                    model_da.values[lat_idx, lon_idx] = val_to_assign

                model_da = model_da.assign_attrs(
                    units="skill_score_percent",
                    model=model,
                    horizon=h,
                    metric=val
                )
                loop_list.append(model_da)

            ret.append(loop_list)  # inside h loop now

    return ret

In [ ]:
rajat_fig6_data_path = "/Users/charlieeden/Downloads/monsoon_cell_scores_2004_2021_ALL_CELLS.csv"
rajat_fig6_df = pd.read_csv(rajat_fig6_data_path)

In [ ]:
lat = np.arange(8, 37, 4)  # 8:4:36
lon = np.arange(68, 101, 4)  # 68:4:100
raja_fig_6_data_grid = package_csv_data_as_grid(
    rajat_fig6_df,
    models_list = ["ifss2s", "fuxis2s", "ngcm"],
    lat=lat,
    lon=lon
)

In [ ]:
def compare_fig6_grids(paper_grid, recreated_grid):
    out = {
        "fair_brier_skill": {15: [], 30: []},
        "fair_rps_skill":   {15: [], 30: []}
    }
    
    for i in range(len(paper_grid)):
        for j in range(len(paper_grid[i])):
            da = paper_grid[i][j]
            score = da.attrs["metric"]
            horizon = da.attrs["horizon"]
            model = da.attrs["model"]
            
            diff = da - recreated_grid[i][j]
            diff.attrs.update({
                "metric": score,
                "horizon": horizon,
                "model": model,
                "units": "skill_score_percent"
            })
            
            out[score][horizon].append(diff)
                
    return out

In [ ]:
test = compare_fig6_grids(raja_fig_6_data_grid, gridded_data)

In [ ]:
def create_fig6_style_diff_figure(
    diff_dict,
    lon,
    lat,
    models,
    shpfile_path,
    cmap="RdBu_r",
):
    row_configs = [
        ("fair_brier_skill", 15, "(a) Brier Skill Score: 15-day forecast"),
        ("fair_brier_skill", 30, "(b) Brier Skill Score: 30-day forecast"),
        ("fair_rps_skill",   15, "(c) Ranked Probability Skill Score: 15-day forecast"),
        ("fair_rps_skill",   30, "(d) Ranked Probability Skill Score: 30-day forecast"),
    ]

    def get_vlims(metric):
        all_das = diff_dict[metric][15] + diff_dict[metric][30]
        vals = np.concatenate([da.values.flatten() for da in all_das])
        vals = vals[~np.isnan(vals)]
        vmax = np.nanpercentile(np.abs(vals), 95)
        return -vmax, vmax

    vlims = {
        "fair_brier_skill": get_vlims("fair_brier_skill"),
        "fair_rps_skill":   get_vlims("fair_rps_skill"),
    }

    fig = plt.figure(figsize=(8, 8), dpi=200)

    gs = GridSpec(
        4, 4, figure=fig,
        hspace=0.2,
        wspace=0.02,
        left=0.08, right=0.88, top=0.95, bottom=0.08,
        width_ratios=[1, 1, 1, 0.06],
        height_ratios=[1, 1, 1, 1],
    )

    axes = []
    for row in range(4):
        row_axes = []
        for col in range(3):
            ax = fig.add_subplot(gs[row, col], projection=ccrs.PlateCarree())
            row_axes.append(ax)
        axes.append(row_axes)

    india_boundaries = get_india_outline(shpfile_path)

    images_bss  = None
    images_rpss = None

    for row_idx, (metric, horizon, row_label) in enumerate(row_configs):
        vmin, vmax = vlims[metric]
        das = diff_dict[metric][horizon]

        for col_idx, da in enumerate(das):
            ax = axes[row_idx][col_idx]

            im = ax.pcolormesh(
                lon, lat, da.values,
                cmap=cmap, vmin=vmin, vmax=vmax,
                transform=ccrs.PlateCarree(),
            )

            if metric == "fair_brier_skill" and images_bss is None:
                images_bss = im
            if metric == "fair_rps_skill" and images_rpss is None:
                images_rpss = im

            for boundary in india_boundaries:
                ilon, ilat = boundary
                ax.plot(ilon, ilat, color="black", linewidth=0.5,
                        transform=ccrs.PlateCarree())

            model_name = da.attrs.get("model", models[col_idx])
            ax.text(0.97, 0.97, model_name,
                    transform=ax.transAxes,
                    ha="right", va="top",
                    fontsize=6.5, color="black")

            ax.set_xlim([lon[0] - 4, lon[-1] + 2])
            ax.set_ylim([lat[0] - 4, lat[-1] + 4])

            yticks = np.arange(np.ceil(lat[0] / 8) * 8, lat[-1] + 1, 8)
            ax.set_yticks(yticks)
            if col_idx == 0:
                ax.set_yticklabels([f"{int(y)}°N" for y in yticks], fontsize=5)
            else:
                ax.set_yticklabels([])

            xticks = np.arange(np.ceil(lon[0] / 8) * 8, lon[-1] + 1, 8)
            ax.set_xticks(xticks)
            if row_idx == 3:
                ax.set_xticklabels([f"{int(x)}°E" for x in xticks], fontsize=5)
            else:
                ax.set_xticklabels([])

            ax.tick_params(length=2, width=0.5)
            ax.grid(False)
            ax.set_aspect("equal", adjustable="box")

        # Row label above leftmost panel
        pos = axes[row_idx][0].get_position()
        fig.text(
            pos.x0,
            pos.y1 + 0.005,
            row_label,
            ha="left", va="bottom",
            fontsize=6.5,
        )

    # BSS colorbar (rows 0-1)
    pos_top    = axes[0][2].get_position()
    pos_bottom = axes[1][2].get_position()
    total_height = pos_top.y1 - pos_bottom.y0
    half_height  = total_height * 0.5
    center_y     = (pos_top.y1 + pos_bottom.y0) / 2
    cax_bss = fig.add_axes([0.85, center_y - half_height / 2, 0.01, half_height])
    cbar_bss = fig.colorbar(images_bss, cax=cax_bss, orientation="vertical", extend="both")
    cbar_bss.set_label("BSS (%)", fontsize=6, rotation=270, labelpad=8)
    cbar_bss.ax.tick_params(labelsize=5, length=2, width=1)
    cbar_bss.ax.minorticks_off()

    # RPSS colorbar (rows 2-3)
    pos_top    = axes[2][2].get_position()
    pos_bottom = axes[3][2].get_position()
    total_height = pos_top.y1 - pos_bottom.y0
    half_height  = total_height * 0.5
    center_y     = (pos_top.y1 + pos_bottom.y0) / 2
    cax_rpss = fig.add_axes([0.85, center_y - half_height / 2, 0.01, half_height])
    cbar_rpss = fig.colorbar(images_rpss, cax=cax_rpss, orientation="vertical", extend="both")
    cbar_rpss.set_label("RPSS (%)", fontsize=6, rotation=270, labelpad=8)
    cbar_rpss.ax.tick_params(labelsize=5, length=2, width=1)
    cbar_rpss.ax.minorticks_off()

    fig.suptitle("Skill Score Differences (paper - recreated)")

    return fig

In [ ]:
test_im = create_fig6_style_diff_figure(
    test,
    lon=lon,
    lat=lat,
    models=["ifss2s", "fuxis2s", "ngcm"],
    shpfile_path = config["shpfile_path"]
)